# COde 03 — Đánh giá nghiên cứu và khóa test

Gắn bank v4, cache notebook01 và output notebook02. Mặc định chỉ đánh giá validation.
Test chỉ mở sau khi hoàn thành thực nghiệm và ghi protocol đã khóa cùng các giới hạn review còn lại.
Không chỉnh phương pháp/hyperparameters theo test.

In [ ]:
from pathlib import Path
import os, sys, json
RESEARCH_BANK_OVERRIDE = None
PROJECT_ROOT_OVERRIDE = None
FEATURE_ROOT_OVERRIDE = None
OUTPUT_ROOT_OVERRIDE = None
CONFIG_FILE = 'research.json'
RUN_NAME = None
SPLIT = 'validation'
TEST_REVIEW_RECORD = None
STUDY_ROOT_OVERRIDE = None
LOCK_TEST_PROTOCOL = False


In [ ]:
from pathlib import Path
import os, sys, json, importlib.util
RESEARCH_BANK_OVERRIDE = globals().get('RESEARCH_BANK_OVERRIDE')
RESEARCH_ASSETS = None

def research_find(predicate, *, include_runs=False):
    found = set()
    for base in (Path.cwd(), Path.cwd().parent / 'Data', Path('/kaggle/input')):
        if not base.exists():
            continue
        for current, dirs, files in os.walk(base):
            here = Path(current)
            if predicate(here, dirs, files):
                found.add(here.resolve())
            excluded = {'Images', '.git', '.cache', '__pycache__', '_archive', '_support',
                        'models', 'wheels', '_chunks', 'student_export'}
            if not include_runs:
                excluded.add('runs')
            dirs[:] = [d for d in dirs if d not in excluded]
            if len(here.relative_to(base).parts) >= 8:
                dirs[:] = []
    return sorted(found, key=str)

def research_one(override, found, name):
    if override is not None:
        return Path(override).resolve()
    if len(found) != 1:
        raise RuntimeError(f'{name}: found {len(found)} folders: {found}. Set the matching *_OVERRIDE.')
    return found[0]

if Path('/kaggle').exists():
    bank_root = research_one(RESEARCH_BANK_OVERRIDE,
        research_find(lambda p, d, f: 'research_bank_manifest.json' in f), 'Research bank')
    helper = bank_root / 'source/_support/research_bank.py'
    spec = importlib.util.spec_from_file_location('research_bank_setup', helper)
    bank = importlib.util.module_from_spec(spec); spec.loader.exec_module(bank)
    RESEARCH_ASSETS = bank.activate(bank_root)
    PROJECT_ROOT = RESEARCH_ASSETS['source_root']
    # Compatibility for existing data-audit publishing cells only.
    OFFLINE_ASSETS = {'root': bank_root, 'source_root': PROJECT_ROOT}
else:
    PROJECT_ROOT = research_one(globals().get('PROJECT_ROOT_OVERRIDE'),
        research_find(lambda p, d, f: (p / 'clindistill/research').is_dir() and 'configs' in d), 'Research source')
    sys.path.insert(0, str(PROJECT_ROOT))
print('Research source:', PROJECT_ROOT)


In [ ]:
from clindistill.artifacts import read_json
FEATURE_ROOT = research_one(FEATURE_ROOT_OVERRIDE,
    research_find(lambda p,d,f: 'feature_manifest.json' in f and
        not read_json(p / 'feature_manifest.json').get('debug') and
        {'train.npz','validation.npz'} <= set(f)), 'Research feature cache')
OUTPUT_ROOT = Path(OUTPUT_ROOT_OVERRIDE) if OUTPUT_ROOT_OVERRIDE else (
    Path('/kaggle/working') if Path('/kaggle/working').exists() else PROJECT_ROOT / '_support')
CONFIG = read_json(PROJECT_ROOT / 'configs/code' / CONFIG_FILE)
RUN_NAME = RUN_NAME or CONFIG['run_name']
CONFIG['run_name'] = RUN_NAME
STUDY_ROOT = OUTPUT_ROOT / 'runs' / RUN_NAME
print('Features:', FEATURE_ROOT, '\nStudy:', STUDY_ROOT)


In [ ]:
from clindistill.research.reporting import evaluate_study, freeze_test_protocol
if STUDY_ROOT_OVERRIDE:
    STUDY_ROOT = Path(STUDY_ROOT_OVERRIDE)
elif not (STUDY_ROOT / 'completed.json').exists():
    STUDY_ROOT = research_one(None, research_find(lambda p,d,f:
        {'study.json','completed.json'} <= set(f) and read_json(p / 'study.json')['config']['run_name'] == RUN_NAME,
        include_runs=True), 'Completed study')
if LOCK_TEST_PROTOCOL and Path('/kaggle/input') in STUDY_ROOT.resolve().parents:
    import shutil
    writable = OUTPUT_ROOT / 'runs' / RUN_NAME
    if writable.exists():
        raise RuntimeError('Set STUDY_ROOT_OVERRIDE to the existing working study before locking.')
    shutil.copytree(STUDY_ROOT, writable)
    STUDY_ROOT = writable
if LOCK_TEST_PROTOCOL:
    protocol = freeze_test_protocol(FEATURE_ROOT, STUDY_ROOT, TEST_REVIEW_RECORD)
    print('Locked:', STUDY_ROOT / 'test_protocol.json', protocol['identity'])
table, summary = evaluate_study(FEATURE_ROOT, STUDY_ROOT, OUTPUT_ROOT / 'reports' / RUN_NAME / SPLIT,
                               split=SPLIT, test_review_record=TEST_REVIEW_RECORD)
display(table); display(summary)


Kết quả có per-class precision/recall/F1, calibration, paired contrasts (có MM KD−Visual KD),
teacher_strata.json cho bốn nhóm Teacher đúng/sai và pending review status.
Đây là phân tích bất đồng mô hình, không chứng minh ca bệnh cần clinical.
CI chưa hiệu chỉnh nhiều so sánh và có điều kiện trên mô hình đã fit.

Lần đầu chỉ SPLIT='validation'. Để khóa test: LOCK_TEST_PROTOCOL=True và TEST_REVIEW_RECORD mô tả thực tế.
Save Version giữ test_protocol.json. Notebook01 dùng file này để tạo test shard trên bản copy cache.
Sau đó notebook03 dùng cache đã thêm test, đúng study đã khóa, SPLIT='test', LOCK_TEST_PROTOCOL=False
và cùng TEST_REVIEW_RECORD. Không tự xóa pending_reviews khi khóa protocol.
